In [227]:
import pandas as pd

In [ ]:
# 스킬 빌더 데이터셋
skill_builder = pd.read_csv('skill_builder_data.csv', encoding='latin1')

/var/folders/m3/ryfyt7vj6b1714rxk43bj2zr0000gn/T/ipykernel_95281/584625153.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  skill_builder = pd.read_csv('OLD/skill_builder_data.csv', encoding='latin1')


In [9]:
# 하나의 문제가 여러 스킬을 가진 경우, 여러 행으로 분리 
skill_builder[skill_builder['problem_id']==93383][['skill_id','skill_name']].drop_duplicates()

,skill_id,skill_name
3957,2.0,Circle Graph
94047,37.0,NaN
172411,70.0,Percent Of


In [ ]:
# 스킬 빌더 데이터셋 전처리

# 필요한 칼럼만 선택
skill_builder = skill_builder[['user_id','order_id','problem_id','correct','sequence_id','skill_id', 'skill_name']]

skill_builder['problem_ID'] = skill_builder['problem_id']
skill_builder['skill_ID'] = skill_builder['skill_id']

# 공백 제거
skill_builder = skill_builder.dropna().sort_values('order_id')

# 필터링: 사용자와 스킬이 충분한 경우만 남김
skill_builder = skill_builder.groupby('user_id').filter(lambda q: len(q) > 5).copy()
skill_builder = skill_builder.groupby('skill_id').filter(lambda q: len(q) > 1).copy()

# 인코딩
skill_builder["skill_id"], skill_labels = pd.factorize(skill_builder["skill_id"])
skill_builder["skill_id"] += 1  

skill_builder["problem_id"], answer_labels = pd.factorize(skill_builder["problem_ID"])
skill_builder["problem_id"] += 1

skill_builder['correct'] = skill_builder['correct'].astype(int)

skill_builder['skill_with_answer'] = skill_builder['skill_id'] * 2 + skill_builder['correct']
skill_builder['problem_with_answer'] = skill_builder['problem_id'] * 2 + skill_builder['correct']

skill_builder.to_csv('OLD/old_skill_builder_data.csv', encoding='utf-8-sig', index=False)

In [ ]:
# 새로 업데이트된 스킬 빌더 데이터셋
df = pd.read_csv('skill_builder_data_corrected_collapsed.csv', encoding='latin1', index_col=0)

# 필요한 칼럼만 선택
df = df[['user_id','order_id','problem_id','correct','sequence_id','skill_id', 'skill_name']]

df['problem_ID'] = df['problem_id']
df['skill_ID'] = df['skill_id']

df = df.dropna().sort_values('order_id')

/var/folders/m3/ryfyt7vj6b1714rxk43bj2zr0000gn/T/ipykernel_95281/1223027303.py:2: DtypeWarning: Columns (17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('OLD/skill_builder_data_corrected_collapsed.csv', encoding='latin1', index_col=0)


In [15]:
df.head(3)

,user_id,order_id,problem_id,correct,sequence_id,skill_id,skill_name,problem_ID,skill_ID
274930,73963,20224085,76429,0,6272,297,Area Trapezoid,76429,297
274931,73963,20224095,76430,1,6272,297,Area Trapezoid,76430,297
274932,73963,20224113,76431,1,6272,297,Area Trapezoid,76431,297


In [17]:
# 한 문제가 여러 스킬을 가진 경우, 여러 스킬의 합으로 구성된 스킬을 구성, 하나의 행으로 처리
df[df['problem_id'].isin(df[df['skill_name']=='Circle Graph']['problem_id'].unique().tolist())]['skill_id'].unique()

# 이 데이터셋은 문제가 있을 가능성...!
# 연달아 같은 문제를 푸는 경우가 많음, 사실상 정답을 알려주는 것이기 때문에, 성능이 좋을 수밖에 없음

array(['2_48_79', '2_70', '2_37_70', '2_37_48_77', '2_37_48'],
      dtype=object)

In [19]:
df = df.groupby('user_id').filter(lambda q: len(q) > 5).copy()
df = df.groupby('skill_id').filter(lambda q: len(q) > 1).copy()

In [20]:
df["skill_id"], skill_labels = pd.factorize(df["skill_id"])
df["skill_id"] += 1  

df["problem_id"], answer_labels = pd.factorize(df["problem_ID"])
df["problem_id"] += 1

df['correct'] = df['correct'].astype(int)

df['skill_with_answer'] = df['skill_id'] * 2 + df['correct']
df['problem_with_answer'] = df['problem_id'] * 2 + df['correct']

df.to_csv('skill_builder_set.csv', encoding='utf-8-sig', index=False)

In [22]:
len(df['skill_id'].unique())

138

In [23]:
len(df['problem_id'].unique())

16865

In [24]:
len(df['user_id'].unique())

3457

____

In [218]:
import os
import json
import pandas as pd

## 아이스크림 데이터셋
## 초등학교 1학년부터 중학교 3학년까지 각 학년의 학습 단위(주제)별로 시행된 온라인 평가 시험지 풀이이력 데이터
## AI 모델이 학생이 어떤 지식태그(Knowledge Tag)를 몇 %의 확률로 아는지 모르는지 추론할 수 있음

## json 파일 dataframe으로 변환


test_root_dir = "Training/[원천]성취수준데이터셋_train/8학년"
test_log = []

for dir_A in os.listdir(test_root_dir): 
    dir_A_path = os.path.join(test_root_dir, dir_A)
    for sub_dir in os.listdir(dir_A_path):
        #sub_dir --> 실력평가n
        sub_dir_path = os.path.join(dir_A_path, sub_dir)
        # 실력평가 하위폴더가 '문항정오답표'로 끝나면
        if sub_dir_path.endswith('1_문항정오답표'):# 실력평가 하위폴더가 '문항정오답표'로 끝나면
            for json_file in os.listdir(sub_dir_path):
                json_path = os.path.join(sub_dir_path, json_file)            
                with open(json_path, 'r') as f:
                    json_content = json.load(f)
                    test_log.append(json_content)


valid_root_dir = "Validation/[원천]성취수준데이터셋_valid/8학년"
valid_log = []

for dir_A in os.listdir(valid_root_dir): 
    dir_A_path = os.path.join(valid_root_dir, dir_A)
    for sub_dir in os.listdir(dir_A_path):
        sub_dir_path = os.path.join(dir_A_path, sub_dir)
        if sub_dir_path.endswith('1_문항정오답표'):
            for json_file in os.listdir(sub_dir_path):
                json_path = os.path.join(sub_dir_path, json_file)            
                with open(json_path, 'r') as f:
                    json_content = json.load(f)
                    valid_log.append(json_content)


problem_root_dir = "Training/[원천]성취수준데이터셋_train/8학년"
problem = []

for dir_A in os.listdir(problem_root_dir): 
    dir_A_path = os.path.join(problem_root_dir, dir_A)
    for sub_dir in os.listdir(dir_A_path):
        sub_dir_path = os.path.join(dir_A_path, sub_dir)
        if sub_dir_path.endswith('2_문항IRT'):
            for json_file in os.listdir(sub_dir_path):
                json_path = os.path.join(sub_dir_path, json_file)            
                with open(json_path, 'r') as f:
                    json_content = json.load(f)
                    problem.append(json_content)

test_log_df = pd.DataFrame(test_log)
valid_log_df = pd.DataFrame(valid_log)

problem_df = pd.DataFrame(problem)

icecream_df = pd.concat([test_log_df, df,valid_log_df])

ice_df = icecream_df[['learnerID','testID','assessmentItemID','answerCode','Timestamp']]

# IRT 3변수: 난이도, 변별도, 추측도
pro_df = problem_df[['testID','assessmentItemID','difficultyLevel','discriminationLevel','guessLevel','knowledgeTag']]

icecream = pd.merge(ice_df, pro_df, on=['testID','assessmentItemID'], how='inner')

#218426
len(icecream)

218426

In [219]:
icecream.rename(columns={'knowledgeTag':'skill_ID','assessmentItemID':'problem_ID',
'learnerID':'user_id','answerCode':'correct'},inplace=True)

icecream = icecream.groupby('user_id').filter(lambda q: len(q) > 5).copy()
icecream = icecream.groupby('skill_ID').filter(lambda q: len(q) > 1).copy()

icecream["skill_id"], skill_labels = pd.factorize(icecream["skill_ID"])
icecream["skill_id"] += 1  

icecream["problem_id"], answer_labels = pd.factorize(icecream["problem_ID"])
icecream["problem_id"] += 1

icecream['correct'] = icecream['correct'].astype(int)

icecream['skill_with_answer'] = icecream['skill_id'] * 2 + icecream['correct']
icecream['problem_with_answer'] = icecream['problem_id'] * 2 + icecream['correct']

In [228]:
with open("수학분야 학습자 역량 측정 데이터/수학지식체계 데이터셋/[라벨]수학 지식체계 데이터 세트_210611.json", "r", encoding="utf-8") as f:
    kc_map = json.load(f)

# 2. Flatten the nested structure
from_id = []
to_id = []
for key, value in kc_map.items():
    from_concept = value["fromConcept"]
    to_concept = value["toConcept"]
    
    from_id.append({
        "kc_id": from_concept["id"],
        "kc_name": from_concept["name"],
        "semester": from_concept["semester"],
        "description": from_concept["description"],
        "chapter": from_concept["chapter"]["name"],
        "achievement": from_concept["achievement"]["name"]})

    to_id.append({    
        "kc_id": to_concept["id"],
        "kc_name": to_concept["name"],
        "semester": to_concept["semester"],
        "description": to_concept["description"],
        "chapter": to_concept["chapter"]["name"],
        "achievement": to_concept["achievement"]["name"]
    })

# 3. Create DataFrame
from_id = pd.DataFrame(from_id)
to_id = pd.DataFrame(to_id)

kc_map_df = pd.concat([from_id, to_id])

In [232]:
kc_map_df['description'].iloc[0]

'임의의 수 $a$와 양의 정수 $n$에 대하여 $a$를 $n$개 거듭하여 곱한 것을 $a$의 $n$제곱이라 하고 $a^n$으로 나타낸다. 또 $a,a^2,a^3,\\cdots,a^n,\\cdots$을 통틀어 $a$의 거듭제곱이라 한다.'

In [233]:
kc_map_df

,kc_id,kc_name,semester,description,chapter,achievement
0,3249,거듭제곱,고등-수1-전체,임의의 수 $a$와 양의 정수 $n$에 대하여 $a$를 $n$개 거듭하여 곱한 것을...,지수함수와 로그함수 > 지수 > 거듭제곱과 거듭제곱근,"거듭제곱과 거듭제곱근의 뜻을 알고, 그 성질을 설명할 수 있다."
1,3,지수가 자연수일 때의 지수법칙,고등-수1-전체,"$a$, $b$가 실수이고 $m$, $n$이 자연수일 때\n\n(1)$a^ma^n=...",지수함수와 로그함수 > 지수 > 거듭제곱과 거듭제곱근,"거듭제곱과 거듭제곱근의 뜻을 알고, 그 성질을 설명할 수 있다."
2,3,지수가 자연수일 때의 지수법칙,고등-수1-전체,"$a$, $b$가 실수이고 $m$, $n$이 자연수일 때\n\n(1)$a^ma^n=...",지수함수와 로그함수 > 지수 > 거듭제곱과 거듭제곱근,"거듭제곱과 거듭제곱근의 뜻을 알고, 그 성질을 설명할 수 있다."
3,3,지수가 자연수일 때의 지수법칙,고등-수1-전체,"$a$, $b$가 실수이고 $m$, $n$이 자연수일 때\n\n(1)$a^ma^n=...",지수함수와 로그함수 > 지수 > 거듭제곱과 거듭제곱근,"거듭제곱과 거듭제곱근의 뜻을 알고, 그 성질을 설명할 수 있다."
4,4,거듭제곱근,고등-수1-전체,방정식 $x^n=a$의 근 $x$를 $a$의 $n$제곱근이라 한다.\n이때 실수 $...,지수함수와 로그함수 > 지수 > 거듭제곱과 거듭제곱근,"거듭제곱과 거듭제곱근의 뜻을 알고, 그 성질을 설명할 수 있다."
...,...,...,...,...,...,...
3441,7268,다섯 자릿수,초등-초4-1학기,"1.다섯 자릿수 알아보기\n10000이 2개, 1000이 3개, 100이 8개, 1...",큰 수 > 다섯 자릿수를 알아볼까요,10000 이상의 큰 수를 읽고 쓸 수 있다.
3442,11269,천만 단위까지의 수 알아보기,초등-초4-1학기,"10000이 1365개인 수- 쓰기: 13560000또는 1356만, 읽기: 천삼백...","큰 수 > 십만, 백만, 천만을 알아볼까요",10000 이상의 큰 수에 대한 자릿값과 위치적 기수법을 원리를 이해한다.
3443,11270,천억 단위까지의 수 알아보기,초등-초4-1학기,1억이 7365개인 수\n- 쓰기: 736500000000또는 7365억\n- 읽기...,큰 수 > 억과 조를 알아볼까요,10000 이상의 큰 수에 대한 자릿값과 위치적 기수법을 원리를 이해한다.
3444,7310,받아 올림이 세 번 있는 $(세자릿수)+(세자릿수)$,초등-초3-1학기,"1. 각 자리의 숫자를 맞추어 적습니다.\n2. 일의 자리, 십의 자리, 백의 자리...",덧셈과 뺄셈 > 덧셈을 해 볼까요 (3),받아 올림이 있는 (세 자릿수)+(세 자릿수)의 계산 원리를 이해하고 그 계산을 할...


In [221]:
## 엣지가 있는 지도이기 때문에 중복이 존재 
print(len(kc_map_df.drop_duplicates()))
print(len(kc_map_df))

1633
6892


In [222]:
kc_map_df = kc_map_df.drop_duplicates()

In [223]:
icecream['kc_id'] = icecream['skill_ID'].astype(int)
data = icecream.merge(kc_map_df, on='kc_id', how='inner')

In [224]:
data = data[['user_id','testID','problem_id','correct','Timestamp', 'difficultyLevel',
       'discriminationLevel','skill_id','skill_with_answer','problem_with_answer',
       'kc_id','kc_name','semester']].sort_values(['user_id','Timestamp'])

In [226]:
#204810
data.to_csv('icecream_8th.csv', encoding='utf-8-sig', index=False)